In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from eda_visualizations import (
    create_univariate_charts,
    survival_by_gender,
    survival_by_class,
    correlation_heatmap,
    multivariate_analysis,
    standardization_check,
    print_survival_rates 
)
def load_dataset():
    """
    Load Titanic dataset once using Seaborn.
    Save an offline copy immediately.
    """

    print("=" * 60)
    print("LOADING TITANIC DATASET")
    print("=" * 60)

    df = sns.load_dataset("titanic")

    # Required by assignment
    df.to_csv("titanic.csv", index=False)

    print("\nDataset loaded successfully.")
    print("Offline dataset saved as 'titanic.csv'.")

    return df


def dataset_profile(df):
    """
    Display dataset profile.
    """

    print("\n" + "=" * 60)
    print("DATASET PROFILE")
    print("=" * 60)

    print("\nShape:")
    print(df.shape)

    print("\nInformation:")
    df.info()

    print("\nDescriptive Statistics:")
    print(df.describe(include="all"))

    print("\nMissing Value Percentages:")

    missing_percentages = (
        df.isnull().sum() / len(df) * 100
    )

    missing_percentages = (
        missing_percentages[missing_percentages > 0]
        .sort_values(ascending=False)
    )

    for column, percentage in missing_percentages.items():
        print(f"{column:<15}: {percentage:.2f}%")

    return missing_percentages


def clean_dataset(df, missing_percentages):
    """
    Apply missing-value handling based on assignment rules.

    <5%      -> Drop rows
    5%–30%   -> Impute
    >30%     -> Drop column
    """

    cleaned_df = df.copy()

    print("\n" + "=" * 60)
    print("MISSING VALUE HANDLING")
    print("=" * 60)

    low_missing = []
    medium_missing = []
    high_missing = []

    # Decide strategy first
    for column, percentage in missing_percentages.items():

        if percentage < 5:
            low_missing.append(column)

        elif percentage <= 30:
            medium_missing.append(column)

        else:
            high_missing.append(column)

    # -----------------------------
    # High Missing (>30%)
    # -----------------------------
    for column in high_missing:

        print(
            f"{column:<15}: "
            f"{missing_percentages[column]:.2f}% -> Column Dropped"
        )

        cleaned_df.drop(columns=column, inplace=True)

    # -----------------------------
    # Medium Missing (5%-30%)
    # -----------------------------
    for column in medium_missing:

        if pd.api.types.is_numeric_dtype(cleaned_df[column]):

            median = cleaned_df[column].median()

            # Warning-free method
            cleaned_df[column] = cleaned_df[column].fillna(median)

            print(
                f"{column:<15}: "
                f"{missing_percentages[column]:.2f}% -> Median Imputation"
            )

        else:

            mode = cleaned_df[column].mode()[0]

            # Warning-free method
            cleaned_df[column] = cleaned_df[column].fillna(mode)

            print(
                f"{column:<15}: "
                f"{missing_percentages[column]:.2f}% -> Mode Imputation"
            )

    # -----------------------------
    # Low Missing (<5%)
    # -----------------------------
    if low_missing:

        before = len(cleaned_df)

        cleaned_df = cleaned_df.dropna(subset=low_missing)

        removed = before - len(cleaned_df)

        print(
            f"{', '.join(low_missing):<25}: "
            f"Rows Dropped ({removed} rows)"
        )

    print("\nCleaned Dataset Shape:")
    print(cleaned_df.shape)

    # Save cleaned dataset
    cleaned_df.to_csv("titanic.csv", index=False)

    print("\nUpdated 'titanic.csv' saved.")

    return cleaned_df

import os
import matplotlib.pyplot as plt
import seaborn as sns



def descriptive_statistics(df):

    print("\n" + "="*60)
    print("DESCRIPTIVE STATISTICS")
    print("="*60)

    for column in ["age", "fare"]:

        print(f"\n{column.upper()}")

        print(f"Mean   : {df[column].mean():.2f}")
        print(f"Median : {df[column].median():.2f}")
        print(f"Mode   : {df[column].mode()[0]:.2f}")


def detect_outliers(df):

    print("\n" + "="*60)
    print("IQR OUTLIER DETECTION")
    print("="*60)

    for column in ["age", "fare"]:

        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers = df[
            (df[column] < lower) |
            (df[column] > upper)
        ]

        print(f"\n{column.upper()}")

        print(f"Lower Limit : {lower:.2f}")
        print(f"Upper Limit : {upper:.2f}")
        print(f"Outliers    : {len(outliers)}")


def skewness_analysis(df):

    print("\n" + "="*60)
    print("SKEWNESS ANALYSIS")
    print("="*60)

    for column in ["age", "fare"]:

        skew = df[column].skew()

        print(f"\n{column.upper()}")

        print(f"Skewness : {skew:.3f}")

        if skew > 1:
            print("Highly Positively Skewed")

        elif skew > 0:
            print("Positively Skewed")

        elif skew < -1:
            print("Highly Negatively Skewed")

        elif skew < 0:
            print("Negatively Skewed")

        else:
            print("Approximately Symmetric")


def main():

    df = load_dataset()

    missing_percentages = dataset_profile(df)

    clean_df = clean_dataset(df, missing_percentages)

    create_univariate_charts(clean_df)

    descriptive_statistics(clean_df)

    detect_outliers(clean_df)

    skewness_analysis(clean_df)

    survival_by_gender(clean_df)

    survival_by_class(clean_df)

    correlation_heatmap(clean_df)

    multivariate_analysis(clean_df)

    standardization_check(clean_df)

    print_survival_rates(clean_df)

    
    

if __name__ == "__main__":
    main()

LOADING TITANIC DATASET

Dataset loaded successfully.
Offline dataset saved as 'titanic.csv'.

DATASET PROFILE

Shape:
(891, 15)

Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), cate